## 1. System Architecture

```
┌─────────────────────┐
│  Restaurant Owner   │
│   (WhatsApp User)   │
└──────────┬──────────┘
           │
           │ Messages
           ▼
┌─────────────────────┐
│ WhatsApp Business   │
│       API           │
└──────────┬──────────┘
           │
           │ Webhook Events
           ▼
┌─────────────────────┐
│  Azure Functions    │
│   (Node.js/GCP)     │
│   - Verify Webhook  │
│   - Store Messages  │
└──────────┬──────────┘
           │
           │ Store/Read
           ▼
┌─────────────────────┐
│     MongoDB         │
│  - Restaurants      │
│  - Strategies       │
│  - Posts            │
│  - MessageLogs      │
└──────────┬──────────┘
           │
           │ Poll/Process
           ▼
┌─────────────────────┐
│  Python Cron Job    │
│  (Jupyter Notebook) │
│   - Process Msgs    │
│   - Send Responses  │
│   - Update States   │
└─────────────────────┘
```

## 2. Conversation Flow & Wireframes

### Main Menu Trigger
**When:** User sends any message while in `IDLE` state or types specific keywords like "menu", "help", "options"

```
┌─────────────────────────────────┐
│  👋 Welcome to RestroPulse!     │
│                                 │
│  What would you like to do?     │
│                                 │
│  [📋 View Business Profile]     │
│  [📅 Check Content Strategy]    │
│  [🖼️  View Next Scheduled Post]  │
│  [💬 Contact Account Manager]   │
└─────────────────────────────────┘
```

**Implementation:** Interactive Button Message (max 3 buttons) + List for 4th option
**State:** `MENU_SELECTION`

### Flow A: View Business Profile

**State Transition:** `MENU_SELECTION` → `IDLE`

```
┌─────────────────────────────────┐
│  🏪 Your Business Profile        │
│                                 │
│  Restaurant: Spice Garden       │
│  Cuisine: Indian, Chinese       │
│  Contact: +91 98765 43210       │
│                                 │
│  📱 Instagram:                  │
│  @spicegarden_mumbai            │
│                                 │
│  💼 Account Manager:            │
│  Priya Sharma                   │
│  📞 +91 98765 11111             │
│                                 │
│  💳 Subscription: Growth Plan   │
│  Status: ✅ Active              │
│                                 │
│  [🏠 Back to Menu]              │
└─────────────────────────────────┘
```

**Implementation:** Text message with formatted info + Quick Reply button

### Flow B: Check Content Strategy

**State Transition:** `MENU_SELECTION` → `AWAITING_STRATEGY_REVIEW`

#### Step 1: Display Current Strategy
```
┌─────────────────────────────────┐
│  📅 Content Strategy             │
│  Cycle: Dec 1-14, 2025          │
│                                 │
│  📝 Summary:                    │
│  Focus on festive season menu,  │
│  customer testimonials, and     │
│  behind-the-scenes content.     │
│                                 │
│  📊 Planned Posts: 12           │
│  - 4 Menu highlights            │
│  - 4 Customer reviews           │
│  - 2 Chef specials              │
│  - 2 Restaurant ambiance        │
│                                 │
│  Status: ⏳ Pending Approval    │
│                                 │
│  [✅ Approve Strategy]          │
│  [📝 Provide Feedback]          │
│  [🏠 Back to Menu]              │
└─────────────────────────────────┘
```

#### Step 2A: If "Provide Feedback" clicked → Launch WhatsApp Flow

**State:** `PROVIDING_STRATEGY_FEEDBACK`

**Flow Screen 1: Feedback Checkboxes**
```
┌─────────────────────────────────┐
│  Strategy Feedback Form          │
│                                 │
│  What needs improvement?        │
│                                 │
│  ☐ Posting frequency            │
│  ☐ Posting timing               │
│  ☐ Content topics               │
│  ☐ Content variety              │
│  ☐ Other                        │
│                                 │
│  [Next →]                       │
└─────────────────────────────────┘
```

**Flow Screen 2: Additional Comments**
```
┌─────────────────────────────────┐
│  Additional Feedback             │
│                                 │
│  ┌─────────────────────────┐   │
│  │ Please share any        │   │
│  │ additional comments...  │   │
│  │                         │   │
│  │                         │   │
│  └─────────────────────────┘   │
│                                 │
│  [Submit Feedback]              │
└─────────────────────────────────┘
```

**After Submission:**
- State: `AWAITING_STRATEGY_REVIEW` → `IDLE`
- Message: "✅ Thank you! Your feedback has been received. Our team will review and update the strategy."

### Flow C: View Next Scheduled Post

**State Transition:** `MENU_SELECTION` → `AWAITING_POST_REVIEW`

#### Step 1: Display Post Preview
```
┌─────────────────────────────────┐
│  🖼️ Next Scheduled Post          │
│                                 │
│  📅 Scheduled: Dec 2, 6:00 PM   │
│                                 │
│  [📸 Image Preview]             │
│  ┌───────────────────────┐     │
│  │                       │     │
│  │   [Post Image]        │     │
│  │                       │     │
│  └───────────────────────┘     │
│                                 │
│  📝 Caption:                    │
│  "Craving something special?    │
│  Try our Chef's Special Paneer  │
│  Tikka! 🔥 Made with love and   │
│  authentic spices. Visit us     │
│  today! #SpiceGarden #Foodie"   │
│                                 │
│  Status: ⏳ Pending Approval    │
│                                 │
│  [✅ Approve Post]              │
│  [📝 Request Changes]           │
│  [🏠 Back to Menu]              │
└─────────────────────────────────┘
```

#### Step 2: If "Request Changes" → Launch Flow

**State:** `PROVIDING_POST_FEEDBACK`

**Flow Screen 1: Feedback Options**
```
┌─────────────────────────────────┐
│  Post Feedback Form              │
│                                 │
│  What needs to change?          │
│                                 │
│  ☐ Image/media content          │
│  ☐ Caption text                 │
│  ☐ Hashtags                     │
│  ☐ Posting time                 │
│  ☐ Other                        │
│                                 │
│  [Next →]                       │
└─────────────────────────────────┘
```

**Flow Screen 2: Detailed Feedback**
```
┌─────────────────────────────────┐
│  Detailed Feedback               │
│                                 │
│  ┌─────────────────────────┐   │
│  │ Please describe what    │   │
│  │ changes you'd like...   │   │
│  │                         │   │
│  │                         │   │
│  └─────────────────────────┘   │
│                                 │
│  [Submit Feedback]              │
└─────────────────────────────────┘
```

### Flow D: Contact Account Manager

**State Transition:** `MENU_SELECTION` → `SUPPORT_QUERY` → `IDLE`

```
┌─────────────────────────────────┐
│  💬 Account Manager Request      │
│                                 │
│  ✅ Request received!           │
│                                 │
│  Your account manager           │
│  Priya Sharma will reach out    │
│  to you within 2 hours.         │
│                                 │
│  For urgent matters, you can    │
│  call: +91 98765 11111          │
│                                 │
│  [🏠 Back to Menu]              │
└─────────────────────────────────┘
```

**Backend Action:**
- Create internal notification/ticket for account manager
- Log the support request with timestamp
- Return to IDLE state

## 3. State Transition Diagram

```
                    ┌──────────┐
                    │   IDLE   │ ◄────────────────┐
                    └────┬─────┘                  │
                         │                        │
         User sends msg  │                        │
         (no context)    │                        │
                         ▼                        │
                  ┌──────────────┐                │
                  │MENU_SELECTION│                │
                  └──────┬───────┘                │
                         │                        │
        ┌────────────────┼────────────────┐       │
        │                │                │       │
        ▼                ▼                ▼       │
  ┌──────────┐  ┌─────────────────┐  ┌──────────────┐
  │ Profile  │  │AWAITING_STRATEGY│  │AWAITING_POST │
  │  View    │  │    _REVIEW      │  │   _REVIEW    │
  └────┬─────┘  └────────┬────────┘  └──────┬───────┘
       │                 │                   │
       │                 ▼                   ▼
       │        ┌─────────────────┐  ┌──────────────┐
       │        │  PROVIDING      │  │  PROVIDING   │
       │        │   STRATEGY      │  │    POST      │
       │        │   FEEDBACK      │  │  FEEDBACK    │
       │        └────────┬────────┘  └──────┬───────┘
       │                 │                   │
       │                 │                   │
       └─────────────────┴───────────────────┘
                         │
                         │ Flow completed/
                         │ Action taken
                         │
                         └───────────────────────────┘
```

## 4. Enhanced MongoDB Schemas

### Key Improvements Needed:

1. **Restaurant Schema** - Add:
   - `pending_bot_response` (Boolean) - Flag for Python cron to process
   - `last_interaction_at` (Date) - Track conversation freshness
   - `contact_number` (String) - Restaurant's phone for profile

2. **Strategy Schema** - Add:
   - `post_breakdown` (Object) - Detailed content type distribution
   - `notification_sent_at` (Date) - When strategy was sent for review

3. **Post Schema** - Already good, but add:
   - `notification_sent_at` (Date) - When post was sent for approval
   - `approved_at` (Date) - Approval timestamp
   - `feedback_tags` (Array) - Similar to Strategy

4. **MessageLog Schema** - Add:
   - `bot_response_sent` (Boolean) - Track if bot replied
   - `context` (String) - What triggered this message (menu_selection, flow_response, etc.)

5. **New Schema: SupportRequest**
   - Track when users request account manager contact
   - `restaurant_id`, `created_at`, `resolved_at`, `notes`

6. **New Schema: WhatsAppFlow**
   - Store Flow IDs, names, and purposes
   - `flow_id`, `flow_name`, `purpose` (strategy_feedback/post_feedback)

## 5. Updated Constants/Enums Needed

### Menu Options
```javascript
const MenuOption = {
    VIEW_PROFILE: 'view_profile',
    CHECK_STRATEGY: 'check_strategy',
    VIEW_NEXT_POST: 'view_next_post',
    CONTACT_MANAGER: 'contact_manager',
    BACK_TO_MENU: 'back_to_menu'
};
```

### Approval Actions
```javascript
const ApprovalAction = {
    APPROVE_STRATEGY: 'approve_strategy',
    FEEDBACK_STRATEGY: 'feedback_strategy',
    APPROVE_POST: 'approve_post',
    FEEDBACK_POST: 'feedback_post'
};
```

### Feedback Tags (for both Strategy & Post)
```javascript
const StrategyFeedbackTag = {
    FREQUENCY: 'posting_frequency',
    TIMING: 'posting_timing',
    TOPICS: 'content_topics',
    VARIETY: 'content_variety',
    OTHER: 'other'
};

const PostFeedbackTag = {
    IMAGE: 'image_content',
    CAPTION: 'caption_text',
    HASHTAGS: 'hashtags',
    TIMING: 'posting_time',
    OTHER: 'other'
};
```

## 6. WhatsApp Message Templates

### Message Types to Use:

1. **Interactive Button Messages** (max 3 buttons)
   - Main menu (first 3 options)
   - Approve/Feedback/Back options

2. **Interactive List Messages** (when 4+ options)
   - Extended menu if needed

3. **WhatsApp Flows** (for complex forms)
   - Strategy feedback form
   - Post feedback form

4. **Media Messages**
   - Send post images for review

5. **Template Messages** (for notifications)
   - "New strategy ready for review"
   - "New post ready for approval"

## 7. Python Cron Job Logic Flow

```python
# Pseudo-code for main processing loop

while True:
    # 1. Find unprocessed messages
    unprocessed_msgs = MessageLog.find({processed: False})
    
    for msg in unprocessed_msgs:
        restaurant = Restaurant.find_one({whatsapp_id: msg.restaurant_whatsapp_id})
        
        # 2. Determine intent based on state + message type
        if restaurant.conversation_state == 'IDLE':
            # Send main menu
            send_main_menu(restaurant)
            restaurant.conversation_state = 'MENU_SELECTION'
            
        elif restaurant.conversation_state == 'MENU_SELECTION':
            # Handle menu button clicks
            if msg.body == MenuOption.VIEW_PROFILE:
                send_business_profile(restaurant)
                restaurant.conversation_state = 'IDLE'
                
            elif msg.body == MenuOption.CHECK_STRATEGY:
                strategy = Strategy.find_one({restaurant_id: restaurant._id, status: 'Pending Approval'})
                send_strategy_review(restaurant, strategy)
                restaurant.conversation_state = 'AWAITING_STRATEGY_REVIEW'
                
            elif msg.body == MenuOption.VIEW_NEXT_POST:
                post = Post.find_one({restaurant_id: restaurant._id, status: 'Pending Approval'}).sort({scheduled_time: 1})
                send_post_review(restaurant, post)
                restaurant.conversation_state = 'AWAITING_POST_REVIEW'
                
            elif msg.body == MenuOption.CONTACT_MANAGER:
                create_support_request(restaurant)
                send_manager_confirmation(restaurant)
                restaurant.conversation_state = 'IDLE'
                
        elif restaurant.conversation_state == 'AWAITING_STRATEGY_REVIEW':
            if msg.body == ApprovalAction.APPROVE_STRATEGY:
                approve_strategy(restaurant)
                send_confirmation("Strategy approved!")
                restaurant.conversation_state = 'IDLE'
                
            elif msg.body == ApprovalAction.FEEDBACK_STRATEGY:
                # This triggers WhatsApp Flow
                restaurant.conversation_state = 'PROVIDING_STRATEGY_FEEDBACK'
                
        elif restaurant.conversation_state == 'PROVIDING_STRATEGY_FEEDBACK':
            if msg.message_type == 'flow_response':
                save_strategy_feedback(restaurant, msg.flow_data)
                send_confirmation("Feedback received!")
                restaurant.conversation_state = 'IDLE'
                
        # ... similar logic for post review states ...
        
        # Mark message as processed
        msg.processed = True
        msg.save()
        restaurant.save()
    
    # Sleep for a few seconds before next poll
    time.sleep(5)
```

## 8. Implementation Priority

### Phase 1: Core Infrastructure (Week 1)
1. ✅ Update MongoDB schemas with new fields
2. ✅ Update webhook constants and enums
3. ✅ Enhance webhook to capture all message types correctly

### Phase 2: Menu System (Week 2)
4. Create main menu interactive message template
5. Implement Flow A (Business Profile)
6. Test menu navigation and state transitions

### Phase 3: Strategy Review (Week 3)
7. Implement Flow B (Strategy Review)
8. Create WhatsApp Flow for strategy feedback
9. Connect feedback to Strategy schema

### Phase 4: Post Approval (Week 4)
10. Implement Flow C (Post Review with media)
11. Create WhatsApp Flow for post feedback
12. Handle approval workflow

### Phase 5: Support & Polish (Week 5)
13. Implement Flow D (Account Manager contact)
14. Add proactive notifications (new strategy/post ready)
15. Error handling and edge cases
16. User testing and refinement

## 9. Next Steps

**Please review the above design and confirm:**

1. ✅ Does the conversation flow make sense?
2. ✅ Are the wireframes clear and user-friendly?
3. ✅ Any changes to the menu options or flows?
4. ✅ Any additional features or scenarios to consider?

**Once approved, I will:**

1. Update the `index.js` webhook with:
   - New constants and enums
   - Enhanced schemas with all new fields
   
2. Create Python notebook with:
   - WhatsApp API integration code
   - Message processing logic for all flows
   - Interactive message builders
   - Flow integration
   
3. Provide WhatsApp Flow JSON definitions for:
   - Strategy feedback form
   - Post feedback form